### Quickstart part
__데이터 작업하기__
- 파이토치에서는 데이터 작업을 위한 기본 요소인 `torch.utils.data.DataLoader`와 `torch.utils.data.Dataset`가 있음
- `Dataset`는 샘플과 정답을 저장, `DataLoader`은 `Dataset`을 순회 가능한 객체로 감쌈

__torchvision.datasets__
- 해당 모듈은 CIFAR, COCO 등과 같은 실제 비전 데이터에 대한 `Dataset`을 포함
- 이번 과정에서는 FasionMNIST 데이터셋을 사용

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [3]:
# 공개 데이터셋 학습 및 테스트 세트 다운로드
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100%|██████████| 26.4M/26.4M [00:04<00:00, 5.71MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 94.8kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 1.78MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 3.50MB/s]


__DataLoader 생성__
- `Dataset`을 `DataLoader`의 인자로 전달
    - 자동화된 배치, 샘플링, 섞기 및 다중 프로세스 데이터 불러오기를 지원
- 배치 사이즈는 64로 지정
    - dataloader는 64개의 특징과 정답을 묶음으로 반환 

In [4]:
batch_size = 64

train_dataloader = DataLoader(training_data, batch_size = batch_size)
test_dataloader = DataLoader(test_data, batch_size = batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


### Model 생성
- PyTorch 에서 신경망 모델은 `nn.Module`을 상속받는 클래스를 생성하여 정의
- `__init__` 함수에서 신경망의 계층을 정의
- `forward` 함수에서 신경망에 데이터를 어떻게 전달할지 지정
    - 가능한 경우 GPU 또는 MPS로 신경망을 이동시켜 가속

In [7]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
        
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
model = NeuralNetwork().to(device)
print(model)        

Using mps device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


### 모델 매개변수 최적화
- 모델을 학습하기 위해 손실함수와 옵티마이저가 필요함
- 모델은 여러번의 epoch를 거쳐서 수행되며, 더 나은 예측을 하기 위해 매개변수를 학습함
    - 각 에폭마다 정확도와 손실을 출력함
- 각 학습 단계에서 모델은 학습 데이터셋에 대한 예측을 수행하고, 예측 오류를 역전파하여 모델의 매개변수를 조정함

In [9]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        
        pred = model(X)
        loss = loss_fn(pred, y)
        
        loss.backward() # 역전파
        optimizer.step()
        optimizer.zero_grad()
        
        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss : {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {100*correct:>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [10]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss : 2.312121  [   64/60000]
loss : 2.295938  [ 6464/60000]
loss : 2.276298  [12864/60000]
loss : 2.263252  [19264/60000]
loss : 2.253401  [25664/60000]
loss : 2.237133  [32064/60000]
loss : 2.228785  [38464/60000]
loss : 2.208230  [44864/60000]
loss : 2.197463  [51264/60000]
loss : 2.163690  [57664/60000]
Test Error: 
 Accuracy: 46.6%, Avg loss: 2.160551 

Epoch 2
-------------------------------
loss : 2.175816  [   64/60000]
loss : 2.158977  [ 6464/60000]
loss : 2.105905  [12864/60000]
loss : 2.115105  [19264/60000]
loss : 2.061435  [25664/60000]
loss : 2.019938  [32064/60000]
loss : 2.033076  [38464/60000]
loss : 1.964615  [44864/60000]
loss : 1.964612  [51264/60000]
loss : 1.892754  [57664/60000]
Test Error: 
 Accuracy: 55.4%, Avg loss: 1.892043 

Epoch 3
-------------------------------
loss : 1.928030  [   64/60000]
loss : 1.887904  [ 6464/60000]
loss : 1.784399  [12864/60000]
loss : 1.822065  [19264/60000]
loss : 1.699808  [25664/60000]
l

### Model 저장 후 불러오기
- 모델을 저장하는 일반적인 방법 -> 모델의 매개변수를 포함하여 Internal state dictionary 직렬화
- 모델을 불러오는 과정 -> 모델 구조를 다시 만들고 state dictionary를 모델에 불러오는 과정이 포함

In [15]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth"))

Saved PyTorch Model State to model.pth


<All keys matched successfully>

- 이제 해당 모델을 사용하여 예측을 수행

In [16]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Dress", Actual: "Ankle boot"
